# Cardiac AI V4 - Danh gia Toan dien & Tim Nguong
**Ap dung chien luoc danh gia chuyen sau tu Chest X-ray code**

3 chien luoc tim nguong:
- **Dang 1 - Youden Index**: TPR - FPR max — can bang gia tri chung
- **Dang 2 - F1 + Recall-constrained**: uu tien F1 nhung dam bao Recall >= 0.75 cho benh nguy hiem
- **Dang 3 - Max Recall (Screening)**: uu tien phat hien benh, chap nhan FP cao hon

Bao gom: ROC Curve, PR Curve, Confusion Matrix, Radar Chart, Heatmap, Bao cao CSV

## Cell 1 - Cai thu vien

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'])
print('Done')

## Cell 2 - Imports & Config

In [ ]:
import os, json, warnings, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

import wfdb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.model_selection import train_test_split

# ── Config (khop voi luc train V4) ────────────────────────────────────────
SEED         = 42
NUM_LEADS    = 12
SEQ_LEN      = 4096
DROPOUT      = 0.15
MS_CHANNELS  = [128, 256, 512, 512]
TRANS_DIM    = 384
TRANS_HEADS  = 8
TRANS_LAYERS = 8
TRANS_FF_DIM = 1536
NUM_CLASSES  = 27
BATCH_SIZE   = 32
NUM_WORKERS  = 4

# ── Benh nguy hiem: uu tien Recall >= MIN_RECALL ──────────────────────────
CRITICAL_DISEASES   = [1, 2, 8, 13, 15, 26]   # AF, AFL, LBBB, PVC, LQT, VPB
MIN_RECALL_CRITICAL = 0.75

DATA_DIR    = ('/kaggle/input/datasets/gamalasran/physionet-challenge-2020/'
               'classification-of-12-lead-ecgs-the-physionetcomputing-in-'
               'cardiology-challenge-2020-1.0.2/training')
MAPPING_CSV = ('/kaggle/input/datasets/bjoernjostein/physionet-snomed-mappings/'
               'SNOMED_mappings_scored.csv')
MODEL_PATH  = '/kaggle/input/datasets/tuyenngoc12/cardiac-ai-v4-model/cardiac_v4_best.pth'
OUTPUT_DIR  = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/thresholds', exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IDX_TO_NAME = {
    0:'IAVB', 1:'AF', 2:'AFL', 3:'Brady', 4:'CRBBB', 5:'IRBBB',
    6:'LAnFB', 7:'LAD', 8:'LBBB', 9:'LQRSV', 10:'NSIVCB', 11:'PR',
    12:'PAC', 13:'PVC', 14:'LPR', 15:'LQT', 16:'QAb', 17:'RAD',
    18:'RBBB', 19:'SA', 20:'SB', 21:'SNR', 22:'STach', 23:'SVPB',
    24:'TAb', 25:'TInv', 26:'VPB'
}
IDX_TO_VIET = {
    0:'Bloc nhi that do 1', 1:'Rung nhi', 2:'Cuong nhi',
    3:'Nhip cham', 4:'Bloc nhanh phai hoan toan',
    5:'Bloc nhanh phai khong hoan toan', 6:'Bloc phan nhanh trai truoc',
    7:'Lech truc trai', 8:'Bloc nhanh trai', 9:'Dien the QRS thap',
    10:'Roi loan dan truyen trong that', 11:'Khoang PR',
    12:'Ngoai tam thu nhi', 13:'Ngoai tam thu that',
    14:'Khoang PR keo dai', 15:'Khoang QT keo dai',
    16:'Song Q bat thuong', 17:'Lech truc phai',
    18:'Bloc nhanh phai', 19:'Roi loan nhip xoang',
    20:'Nhip cham xoang', 21:'Nhip xoang binh thuong',
    22:'Nhip nhanh xoang', 23:'Nhip som tren that',
    24:'Song T bat thuong', 25:'Song T dao nguoc', 26:'Nhip som that'
}
print(f'Device: {device}')
print('Config OK')

## Cell 3 - Load Data & Model

In [ ]:
# ── SNOMED ────────────────────────────────────────────────────────────────
df_map = pd.read_csv(MAPPING_CSV, sep=';')
SNOMED_TO_IDX = {}
for idx, row in df_map.iterrows():
    try: SNOMED_TO_IDX[int(row['SNOMED CT Code'])] = idx
    except: pass

def parse_hea_labels(hea_path):
    labels = []
    try:
        with open(hea_path, 'r', errors='ignore') as fh:
            for line in fh:
                if line.startswith('#') and 'Dx:' in line:
                    for tok in line.split('Dx:')[-1].split(','):
                        tok = tok.strip()
                        if tok.isdigit() and int(tok) in SNOMED_TO_IDX:
                            labels.append(SNOMED_TO_IDX[int(tok)])
    except: pass
    return list(set(labels))

print('Scanning files...')
records = []
for root, _, files in os.walk(DATA_DIR):
    for fname in files:
        if not fname.endswith('.hea'): continue
        hea = os.path.join(root, fname)
        mat = hea.replace('.hea', '.mat')
        if not os.path.exists(mat): continue
        lbls = parse_hea_labels(hea)
        if lbls: records.append({'hea_path':hea,'mat_path':mat,'labels':lbls})

df_records   = pd.DataFrame(records)
label_matrix = np.zeros((len(df_records), NUM_CLASSES), dtype=np.float32)
for i, row in df_records.iterrows():
    for lbl in row['labels']:
        if lbl < NUM_CLASSES: label_matrix[i, lbl] = 1.0

idx_all = np.arange(len(df_records))
train_idx, tmp_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED)
val_idx,  test_idx = train_test_split(tmp_idx,  test_size=0.50, random_state=SEED)
print(f'Records: {len(df_records):,} | Test: {len(test_idx):,} | Val: {len(val_idx):,}')

# ── Dataset ───────────────────────────────────────────────────────────────
class ECGDataset(Dataset):
    def __init__(self, df, lm, seq_len=SEQ_LEN):
        self.df=df.reset_index(drop=True); self.orig_idx=df.index.to_numpy()
        self.lm=lm; self.seq_len=seq_len
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        try:
            rec=wfdb.rdrecord(self.df.iloc[idx]['mat_path'].replace('.mat',''))
            sig=rec.p_signal.astype(np.float32)
        except: sig=np.zeros((self.seq_len,NUM_LEADS),dtype=np.float32)
        mu=sig.mean(0,keepdims=True); std=sig.std(0,keepdims=True)+1e-8; sig=(sig-mu)/std
        T=sig.shape[0]
        if T>=self.seq_len:
            s=(T-self.seq_len)//2; sig=sig[s:s+self.seq_len]
        else: sig=np.vstack([sig,np.zeros((self.seq_len-T,NUM_LEADS),np.float32)])
        return torch.from_numpy(sig.T), torch.from_numpy(self.lm[self.orig_idx[idx]].copy())

# ── Model V4 ──────────────────────────────────────────────────────────────
class MultiScaleConvBlock(nn.Module):
    def __init__(self,in_ch,out_ch,pool=2):
        super().__init__(); b=out_ch//4
        self.branch_small=nn.Sequential(nn.Conv1d(in_ch,b,3,padding=1,bias=False),nn.GroupNorm(min(8,b),b),nn.GELU())
        self.branch_mid  =nn.Sequential(nn.Conv1d(in_ch,b,9,padding=4,bias=False),nn.GroupNorm(min(8,b),b),nn.GELU())
        self.branch_large=nn.Sequential(nn.Conv1d(in_ch,b,19,padding=9,bias=False),nn.GroupNorm(min(8,b),b),nn.GELU())
        self.branch_pool =nn.Sequential(nn.AvgPool1d(3,stride=1,padding=1),nn.Conv1d(in_ch,b,1,bias=False),nn.GroupNorm(min(8,b),b),nn.GELU())
        self.fusion=nn.Sequential(nn.Conv1d(out_ch,out_ch,3,padding=1,bias=False),nn.GroupNorm(min(8,out_ch),out_ch),nn.GELU())
        self.pool=nn.MaxPool1d(pool)
        self.shortcut=(nn.Sequential(nn.Conv1d(in_ch,out_ch,1,bias=False),nn.GroupNorm(min(8,out_ch),out_ch)) if in_ch!=out_ch else nn.Identity())
    def forward(self,x):
        return self.pool(self.fusion(torch.cat([self.branch_small(x),self.branch_mid(x),self.branch_large(x),self.branch_pool(x)],1))+self.shortcut(x))

class PositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=512,dropout=DROPOUT):
        super().__init__(); self.dropout=nn.Dropout(dropout)
        pe=torch.zeros(max_len,d_model); pos=torch.arange(max_len).unsqueeze(1).float()
        div=torch.exp(torch.arange(0,d_model,2).float()*(-np.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self,x): return self.dropout(x+self.pe[:,:x.size(1)])

class ECGModelV4(nn.Module):
    def __init__(self,num_classes=NUM_CLASSES):
        super().__init__()
        ch=[NUM_LEADS]+MS_CHANNELS
        self.cnn=nn.Sequential(*[MultiScaleConvBlock(ch[i],ch[i+1]) for i in range(len(MS_CHANNELS))])
        self.proj=nn.Sequential(nn.Linear(MS_CHANNELS[-1],TRANS_DIM),nn.LayerNorm(TRANS_DIM))
        self.pos_enc=PositionalEncoding(TRANS_DIM,max_len=1024)
        enc=nn.TransformerEncoderLayer(d_model=TRANS_DIM,nhead=TRANS_HEADS,dim_feedforward=TRANS_FF_DIM,
            dropout=DROPOUT,activation='gelu',batch_first=True,norm_first=True)
        self.transformer=nn.TransformerEncoder(enc,num_layers=TRANS_LAYERS)
        self.head=nn.Sequential(nn.LayerNorm(TRANS_DIM),nn.Dropout(DROPOUT),
                                  nn.Linear(TRANS_DIM,512),nn.GELU(),nn.Dropout(DROPOUT/2),nn.Linear(512,num_classes))
    def forward(self,x):
        x=self.cnn(x); x=x.permute(0,2,1); x=self.proj(x)
        x=self.pos_enc(x); x=self.transformer(x); x=x.mean(1)
        return self.head(x)

model=ECGModelV4().to(device)
ckpt=torch.load(MODEL_PATH,map_location=device)
model.load_state_dict(ckpt['state'])
model.eval()
print(f'Model V4 loaded! epoch={ckpt["epoch"]} val_F1={ckpt["score"]:.4f}')
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

## Cell 4 - Predict TTA 5x (Val + Test)

In [ ]:
def predict_tta(loader, n_tta=5, split_name='SET'):
    print(f'TTA {n_tta}x [{split_name}]...')
    t0 = time.time()

    # Round 1: clean (khong augment)
    all_probs, all_targets = [], []
    with torch.no_grad():
        for x, labels in tqdm(loader, leave=False, desc='Clean predict'):
            logits = model(x.to(device))
            all_probs.append(torch.sigmoid(logits).float().cpu().numpy())
            all_targets.append(labels.numpy())
    probs  = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)

    # TTA rounds: augment nhe
    for t in range(n_tta):
        aug_probs = []
        with torch.no_grad():
            for x, _ in tqdm(loader, leave=False, desc=f'TTA {t+1}/{n_tta}'):
                x = x + torch.randn_like(x) * 0.03
                x = x * (0.9 + torch.rand(x.shape[0],1,1) * 0.2)
                logits = model(x.to(device))
                aug_probs.append(torch.sigmoid(logits).float().cpu().numpy())
        probs += np.concatenate(aug_probs)

    probs /= (n_tta + 1)
    elapsed = time.time() - t0
    print(f'  Done! {y_true.shape[0]:,} samples | {elapsed/60:.1f} min')
    return probs, y_true

val_ds   = ECGDataset(df_records.iloc[val_idx],  label_matrix)
test_ds  = ECGDataset(df_records.iloc[test_idx], label_matrix)
val_loader  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Predict tren ca Val (de tim nguong) va Test (de danh gia cuoi)
y_prob_val,  y_true_val  = predict_tta(val_loader,  n_tta=5, split_name='VAL')
y_prob_test, y_true_test = predict_tta(test_loader, n_tta=5, split_name='TEST')

print(f'Val  : {y_prob_val.shape}  |  Test: {y_prob_test.shape}')

## Cell 5 - 3 Chien luoc Tim Nguong (tren Val set)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DANG 1 — YOUDEN INDEX  (TPR - FPR max)
#   Hoc tu: _strategy_youden trong Chest X-ray code
# ═══════════════════════════════════════════════════════════════════════════
def strategy_youden(y_t, y_p):
    try:
        fpr, tpr, thrs = roc_curve(y_t, y_p)
        idx  = np.argmax(tpr - fpr)
        thr  = float(np.clip(thrs[idx], 0.05, 0.95))
        youd = float((tpr - fpr)[idx])
    except:
        thr, youd = 0.5, 0.0
    return thr, youd

# ═══════════════════════════════════════════════════════════════════════════
# DANG 2 — F1 + RECALL CONSTRAINT  (benh nguy hiem: recall >= 0.75)
#   Hoc tu: _strategy_advanced trong Chest X-ray code
#   Fallback: neu khong dat recall constraint → relax ve thr=0.05
# ═══════════════════════════════════════════════════════════════════════════
def strategy_f1_recall(y_t, y_p, ci):
    is_critical = ci in CRITICAL_DISEASES
    best_f1, best_thr, best_rec = -1, 0.5, 0.0
    for thr in np.arange(0.05, 0.96, 0.01):
        y_b  = (y_p >= thr).astype(int)
        f1   = f1_score(y_t, y_b, zero_division=0)
        tp   = ((y_b==1)&(y_t==1)).sum()
        fn   = ((y_b==0)&(y_t==1)).sum()
        rec  = tp / (tp + fn + 1e-9)
        if is_critical and rec < MIN_RECALL_CRITICAL:
            continue
        if f1 > best_f1:
            best_f1, best_thr, best_rec = f1, float(thr), float(rec)
    # Fallback khi benh nguy hiem khong dat recall constraint
    if best_f1 == -1 and is_critical:
        best_thr = 0.05
        y_b      = (y_p >= best_thr).astype(int)
        best_f1  = f1_score(y_t, y_b, zero_division=0)
        tp = ((y_b==1)&(y_t==1)).sum(); fn = ((y_b==0)&(y_t==1)).sum()
        best_rec = tp/(tp+fn+1e-9)
        print(f'  [Fallback] ci={ci} {IDX_TO_NAME.get(ci,"?")} khong dat recall>={MIN_RECALL_CRITICAL} → thr=0.05 recall={best_rec:.3f}')
    return best_thr, best_f1, best_rec

# ═══════════════════════════════════════════════════════════════════════════
# DANG 3 — MAX RECALL SCREENING  (f1 >= 0.05)
#   Hoc tu: _strategy_screening trong Chest X-ray code
# ═══════════════════════════════════════════════════════════════════════════
def strategy_screening(y_t, y_p):
    best_rec, best_thr, best_f1 = -1, 0.3, 0.0
    for thr in np.arange(0.05, 0.96, 0.01):
        y_b = (y_p >= thr).astype(int)
        f1  = f1_score(y_t, y_b, zero_division=0)
        if f1 < 0.05: continue
        tp  = ((y_b==1)&(y_t==1)).sum(); fn = ((y_b==0)&(y_t==1)).sum()
        rec = tp/(tp+fn+1e-9)
        if rec > best_rec:
            best_rec, best_thr, best_f1 = float(rec), float(thr), float(f1)
    return best_thr, best_rec, best_f1

# ═══════════════════════════════════════════════════════════════════════════
# THUC THI — tim nguong tren VAL set
# ═══════════════════════════════════════════════════════════════════════════
print(f'{"="*78}')
print(f'TIM NGUONG TOI UU TREN VAL SET — 3 CHIEN LUOC')
print(f'{"="*78}')
print(f'{"Idx":>4} {"Ten":>8} {"Youden":>8} {"F1+Rec":>8} {"Screen":>8} {"n_val":>7}  ⚠=nguy hiem')
print(f'{"-"*78}')

thr_youden, thr_f1rec, thr_screen = [], [], []
thr_details = []

for ci in range(NUM_CLASSES):
    n_pos = int(y_true_val[:, ci].sum())
    name  = IDX_TO_NAME.get(ci,'?')
    flag  = '⚠' if ci in CRITICAL_DISEASES else ''

    if n_pos == 0:
        thr_youden.append(0.5); thr_f1rec.append(0.5); thr_screen.append(0.5)
        thr_details.append({'ci':ci,'name':name,'thr_youden':0.5,'thr_f1rec':0.5,'thr_screen':0.5,'n_val':0})
        continue

    ty, youd       = strategy_youden(y_true_val[:,ci], y_prob_val[:,ci])
    tf, f1v, recv  = strategy_f1_recall(y_true_val[:,ci], y_prob_val[:,ci], ci)
    ts, recs, f1s  = strategy_screening(y_true_val[:,ci], y_prob_val[:,ci])

    thr_youden.append(ty); thr_f1rec.append(tf); thr_screen.append(ts)
    thr_details.append({'ci':ci,'name':name,'thr_youden':ty,'thr_f1rec':tf,'thr_screen':ts,'n_val':n_pos})
    print(f'{ci:>4} {name:>8} {ty:>8.3f} {tf:>8.3f} {ts:>8.3f} {n_pos:>7}  {flag}')

print(f'{"="*78}')
print(f'Benh nguy hiem (ci={CRITICAL_DISEASES}): {[IDX_TO_NAME[i] for i in CRITICAL_DISEASES]}')
print(f'Recall constraint: >= {MIN_RECALL_CRITICAL}')

## Cell 6 - Danh gia tong hop tren Test set

In [ ]:
# ── Ham tinh day du metrics (hoc tu compute_metrics trong Chest X-ray) ────
def compute_metrics(y_true, y_prob, thresholds, tag=''):
    records = []
    for ci in range(NUM_CLASSES):
        thr  = thresholds[ci]
        y_t  = y_true[:,ci]; y_p  = y_prob[:,ci]
        y_b  = (y_p >= thr).astype(int)
        n_pos = int(y_t.sum())
        if n_pos == 0:
            records.append({'class':ci,'name':IDX_TO_NAME.get(ci,'?'),
                             'viet':IDX_TO_VIET.get(ci,'?'),'n_test':0,'threshold':thr,
                             'f1':0,'precision':0,'recall':0,'specificity':0,
                             'auc':np.nan,'ap':np.nan,'tp':0,'fp':0,'tn':0,'fn':0})
            continue
        tn,fp,fn,tp = confusion_matrix(y_t, y_b, labels=[0,1]).ravel()
        sens = tp/(tp+fn+1e-9); spec = tn/(tn+fp+1e-9)
        prec = tp/(tp+fp+1e-9); f1   = 2*prec*sens/(prec+sens+1e-9)
        try:    auc = roc_auc_score(y_t, y_p)
        except: auc = np.nan
        try:    ap  = average_precision_score(y_t, y_p)
        except: ap  = np.nan
        records.append({'class':ci,'name':IDX_TO_NAME.get(ci,'?'),
                         'viet':IDX_TO_VIET.get(ci,'?'),'n_test':n_pos,
                         'threshold':round(thr,3),'tp':int(tp),'fp':int(fp),
                         'tn':int(tn),'fn':int(fn),'f1':round(f1,4),
                         'precision':round(prec,4),'recall':round(sens,4),
                         'specificity':round(spec,4),'auc':round(auc,4),'ap':round(ap,4)})
    return pd.DataFrame(records)

def summary_metrics(y_true, y_prob, thresholds):
    mask   = y_true.sum(0) > 0
    y_pred = (y_prob >= np.array(thresholds)).astype(int)
    macro_f1 = f1_score(y_true, y_pred, average='macro',  zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro',  zero_division=0)
    try:    auc = roc_auc_score(y_true[:,mask], y_prob[:,mask], average='macro')
    except: auc = 0.0
    try:    mAP = average_precision_score(y_true[:,mask], y_prob[:,mask], average='macro')
    except: mAP = 0.0
    return {'macro_f1':macro_f1,'micro_f1':micro_f1,'auc':auc,'mAP':mAP}

# ── Danh gia 3 dang ────────────────────────────────────────────────────────
df_d1 = compute_metrics(y_true_test, y_prob_test, thr_youden, 'Dang 1 Youden')
df_d2 = compute_metrics(y_true_test, y_prob_test, thr_f1rec,  'Dang 2 F1+Recall')
df_d3 = compute_metrics(y_true_test, y_prob_test, thr_screen, 'Dang 3 Screening')

s1 = summary_metrics(y_true_test, y_prob_test, thr_youden)
s2 = summary_metrics(y_true_test, y_prob_test, thr_f1rec)
s3 = summary_metrics(y_true_test, y_prob_test, thr_screen)

metrics_dict = {'D1-Youden':df_d1, 'D2-F1+Recall':df_d2, 'D3-Screening':df_d3}

# ── Bang so sanh tong hop (hoc tu print_comparison_table) ─────────────────
print(f'{"="*90}')
print(f'BANG SO SANH 3 CHIEN LUOC — TAP TEST (TTA 5x)')
print(f'{"="*90}')
print(f'{"Ten":>8}  {"AUC_D1":>8}{"F1_D1":>7}{"Rec_D1":>8}  {"AUC_D2":>8}{"F1_D2":>7}{"Rec_D2":>8}  {"AUC_D3":>8}{"F1_D3":>7}{"Rec_D3":>8}')
print(f'{"-"*90}')

for ci in range(NUM_CLASSES):
    if df_d1[df_d1['class']==ci]['n_test'].values[0] == 0: continue
    n  = IDX_TO_NAME.get(ci,'?')
    r1 = df_d1[df_d1['class']==ci].iloc[0]
    r2 = df_d2[df_d2['class']==ci].iloc[0]
    r3 = df_d3[df_d3['class']==ci].iloc[0]
    flag = ' ⚠' if ci in CRITICAL_DISEASES else ''
    print(f'{n:>8}{flag}  {r1.auc:>8.3f}{r1.f1:>7.3f}{r1.recall:>8.3f}  {r2.auc:>8.3f}{r2.f1:>7.3f}{r2.recall:>8.3f}  {r3.auc:>8.3f}{r3.f1:>7.3f}{r3.recall:>8.3f}')

print(f'{"-"*90}')
print(f'{"MEAN":>8}   {df_d1.auc.mean():>8.4f}{df_d1.f1.mean():>7.4f}{df_d1.recall.mean():>8.4f}  {df_d2.auc.mean():>8.4f}{df_d2.f1.mean():>7.4f}{df_d2.recall.mean():>8.4f}  {df_d3.auc.mean():>8.4f}{df_d3.f1.mean():>7.4f}{df_d3.recall.mean():>8.4f}')
print(f'{"="*90}')

print(f'\n{"="*55}')
print(f'TONG QUAN THEO CHIEN LUOC')
print(f'{"="*55}')
for tag, s in [('D1-Youden',s1),('D2-F1+Recall',s2),('D3-Screening',s3)]:
    print(f'  {tag:<16} MacroF1={s["macro_f1"]:.4f} AUC={s["auc"]:.4f} mAP={s["mAP"]:.4f}')
print(f'{"="*55}')

## Cell 7 - Radar Chart & Heatmap so sanh 3 chien luoc

In [ ]:
# ── Radar Chart (hoc tu plot_radar trong Chest X-ray) ─────────────────────
fig = plt.figure(figsize=(16, 7), facecolor='#0f172a')

# 7a: Radar Chart
categories = ['Macro F1','Recall','Specificity','AUC','Precision']
n_cat      = len(categories)
angles     = [x/n_cat*2*np.pi for x in range(n_cat)] + [0]
ax_rad = fig.add_subplot(121, polar=True)
ax_rad.set_facecolor('#1e293b')

colors = ['#22d3ee','#22c55e','#f59e0b']
dang_labels = ['D1-Youden','D2-F1+Recall','D3-Screening']
for df_ev, color, label in zip([df_d1,df_d2,df_d3], colors, dang_labels):
    df_v = df_ev[df_ev.n_test>0]
    vals = [
        df_v.f1.mean(), df_v.recall.mean(), df_v.specificity.mean(),
        df_v.auc.mean(), df_v.precision.mean()
    ] + [df_v.f1.mean()]
    ax_rad.plot(angles, vals, color=color, lw=2.5, label=label)
    ax_rad.fill(angles, vals, color=color, alpha=0.08)

ax_rad.set_xticks(angles[:-1])
ax_rad.set_xticklabels(categories, color='#94a3b8', fontsize=9)
ax_rad.set_ylim(0,1); ax_rad.set_yticks([0.2,0.4,0.6,0.8,1.0])
ax_rad.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], color='#475569', fontsize=7)
ax_rad.set_title('So sanh Mean Metrics 3 chien luoc', color='white', fontsize=11, fontweight='bold', pad=18)
ax_rad.legend(loc='upper right', bbox_to_anchor=(1.35,1.1), facecolor='#1e293b', labelcolor='white', fontsize=8)
ax_rad.grid(True, alpha=0.2, color='#334155')

# 7b: Heatmap Recall (hoc tu plot_heatmap trong Chest X-ray)
ax_hm = fig.add_subplot(122)
ax_hm.set_facecolor('#0f172a')
df_valid = df_d1[df_d1.n_test>0]
recall_data = pd.DataFrame({
    'D1-Youden':  df_d1[df_d1.n_test>0].set_index('name').recall,
    'D2-F1+Rec':  df_d2[df_d2.n_test>0].set_index('name').recall,
    'D3-Screen':  df_d3[df_d3.n_test>0].set_index('name').recall,
})
sns.heatmap(recall_data, ax=ax_hm, annot=True, fmt='.2f', cmap='YlOrRd',
            vmin=0, vmax=1, linewidths=0.5, annot_kws={'size':7,'weight':'bold'})
ax_hm.set_title('Recall (Sensitivity) tung benh x Chien luoc', color='white', fontsize=10, fontweight='bold')
ax_hm.set_xlabel('Chien luoc', color='#94a3b8')
ax_hm.set_ylabel('')
ax_hm.tick_params(axis='y', labelsize=7, colors='#94a3b8')
ax_hm.tick_params(axis='x', labelsize=8, colors='#94a3b8')

plt.suptitle('Cardiac AI V4 - So sanh 3 Chien luoc Nguong', color='white',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/radar_heatmap_v4.png', dpi=120,
            bbox_inches='tight', facecolor='#0f172a')
plt.show(); plt.close()
print('Saved: radar_heatmap_v4.png')

## Cell 8 - ROC Curve & PR Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22,9), facecolor='#0f172a')
colors = plt.cm.rainbow(np.linspace(0,1,NUM_CLASSES))

# ROC (hoc tu plot_final_roc trong Chest X-ray)
ax = axes[0]; ax.set_facecolor('#0f172a')
auc_scores = []
for ci, color in zip(range(NUM_CLASSES), colors):
    if y_true_test[:,ci].sum()==0: continue
    try:
        fpr,tpr,_ = roc_curve(y_true_test[:,ci], y_prob_test[:,ci])
        auc = roc_auc_score(y_true_test[:,ci], y_prob_test[:,ci])
        auc_scores.append(auc)
        lw  = 2.5 if ci in CRITICAL_DISEASES else 1.2
        ax.plot(fpr, tpr, color=color, lw=lw, label=f'{IDX_TO_NAME[ci]} ({auc:.3f})')
    except: pass

ax.plot([0,1],[0,1],'w--',lw=1.5,alpha=0.5,label='Random (0.500)')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate (1 - Specificity)', color='#94a3b8', fontsize=11)
ax.set_ylabel('True Positive Rate (Sensitivity)', color='#94a3b8', fontsize=11)
ax.set_title(f'ROC Curves — Mean AUC={np.mean(auc_scores):.4f}\n(굵은net = benh nguy hiem)',
              color='white', fontsize=12, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01,1), loc='upper left', fontsize=6.5,
           facecolor='#1e293b', labelcolor='white')
ax.tick_params(colors='#475569'); ax.grid(alpha=0.15, color='#334155')
for sp in ax.spines.values(): sp.set_color('#1e293b')

# PR Curve (hoc tu plot_pr_curves trong Chest X-ray)
ax = axes[1]; ax.set_facecolor('#0f172a')
ap_scores = []
for ci, color in zip(range(NUM_CLASSES), colors):
    if y_true_test[:,ci].sum()==0: continue
    try:
        prec_arr,rec_arr,_ = precision_recall_curve(y_true_test[:,ci], y_prob_test[:,ci])
        ap = average_precision_score(y_true_test[:,ci], y_prob_test[:,ci])
        ap_scores.append(ap)
        lw = 2.5 if ci in CRITICAL_DISEASES else 1.2
        ax.plot(rec_arr, prec_arr, color=color, lw=lw, label=f'{IDX_TO_NAME[ci]} (AP={ap:.3f})')
    except: pass

ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('Recall', color='#94a3b8', fontsize=11)
ax.set_ylabel('Precision', color='#94a3b8', fontsize=11)
ax.set_title(f'PR Curves — mAP={np.mean(ap_scores):.4f}\n(PR quan trong hon ROC khi class imbalance)',
              color='white', fontsize=12, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01,1), loc='upper left', fontsize=6.5,
           facecolor='#1e293b', labelcolor='white')
ax.tick_params(colors='#475569'); ax.grid(alpha=0.15, color='#334155')
for sp in ax.spines.values(): sp.set_color('#1e293b')

plt.suptitle('Cardiac AI V4 - ROC & PR Curves (Test set, TTA 5x)',
              color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/roc_pr_v4.png', dpi=120, bbox_inches='tight', facecolor='#0f172a')
plt.show(); plt.close()
print('Saved: roc_pr_v4.png')

## Cell 9 - Confusion Matrix tung benh (27 benh)

In [ ]:
def plot_confusion_matrices(y_true, y_pred_b, title_tag='', facecolor='#0f172a'):
    '''Hoc tu plot_confusion_matrices trong Chest X-ray code'''
    valid_classes = [ci for ci in range(NUM_CLASSES) if y_true[:,ci].sum()>0]
    n  = len(valid_classes)
    nc = 6; nr = (n + nc - 1) // nc
    fig, axes = plt.subplots(nr, nc, figsize=(22, nr*3.8), facecolor=facecolor)
    axes = axes.flatten()
    fig.suptitle(f'Confusion Matrix — {title_tag}', color='white', fontsize=13, fontweight='bold', y=1.01)

    for ax_i, ci in enumerate(valid_classes):
        cm             = confusion_matrix(y_true[:,ci], y_pred_b[:,ci], labels=[0,1])
        tn,fp,fn,tp    = cm.ravel()
        cm_norm        = cm.astype(float).copy()
        if (tn+fp)>0: cm_norm[0] = cm_norm[0]/(tn+fp)
        if (fn+tp)>0: cm_norm[1] = cm_norm[1]/(fn+tp)
        ax = axes[ax_i]
        ax.set_facecolor('#1e293b')
        im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
        flag = ' ⚠' if ci in CRITICAL_DISEASES else ''
        ax.set_title(f'{IDX_TO_NAME.get(ci,"?")}{flag}', fontsize=9, fontweight='bold', color='white')
        ax.set_xticks([0,1]); ax.set_yticks([0,1])
        ax.set_xticklabels(['Pred(-)','Pred(+)'], fontsize=7, color='#94a3b8')
        ax.set_yticklabels(['True(-)','True(+)'], fontsize=7, color='#94a3b8')
        for row in range(2):
            for col in range(2):
                v_raw=cm[row,col]; v_norm=cm_norm[row,col]
                color='white' if v_norm>0.6 else 'black'
                ax.text(col, row, f'{v_raw}\n({v_norm:.0%})',
                         ha='center', va='center', fontsize=7, color=color, fontweight='bold')
        for sp in ax.spines.values(): sp.set_color('#334155')

    for ax_i in range(len(valid_classes), len(axes)):
        axes[ax_i].set_visible(False)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/confusion_{title_tag.replace(" ","_")}.png',
                dpi=100, bbox_inches='tight', facecolor=facecolor)
    plt.show(); plt.close()
    print(f'Saved: confusion_{title_tag.replace(" ","_")}.png')

# Ve Confusion Matrix cho D1 (Youden) — dang chuan
y_pred_d1 = (y_prob_test >= np.array(thr_youden)).astype(int)
y_pred_d2 = (y_prob_test >= np.array(thr_f1rec)).astype(int)

plot_confusion_matrices(y_true_test, y_pred_d1, title_tag='D1 Youden')
plot_confusion_matrices(y_true_test, y_pred_d2, title_tag='D2 F1+Recall')

## Cell 10 - Bieu do F1 / Recall / Specificity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 9), facecolor='#0f172a')
df_plot = df_d2[df_d2.n_test>0].sort_values('f1', ascending=True)

def bar_h(ax, values, labels, title, xlabel, vline=None, cmap_fn=None):
    ax.set_facecolor('#0f172a')
    if cmap_fn is None:
        colors = ['#22c55e' if v>=0.7 else '#f59e0b' if v>=0.5 else '#ef4444' for v in values]
    else:
        colors = [cmap_fn(v) for v in values]
    ax.barh(range(len(labels)), values, color=colors, height=0.72)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, color='#94a3b8', fontsize=8)
    ax.set_xlabel(xlabel, color='#94a3b8'); ax.set_title(title, color='white', fontweight='bold', fontsize=10)
    ax.set_xlim(0,1.12)
    if vline:
        ax.axvline(x=vline, color='white', linestyle='--', alpha=0.6, lw=1.5)
        ax.text(vline+0.01, len(labels)-1, f'{vline:.3f}', color='white', fontsize=7)
    ax.tick_params(colors='#475569'); ax.grid(axis='x', alpha=0.15, color='#334155')
    for sp in ax.spines.values(): sp.set_color('#1e293b')

names  = df_plot.name.tolist()
recall_cmap = lambda v: '#22c55e' if v>=0.8 else '#f59e0b' if v>=MIN_RECALL_CRITICAL else '#ef4444'
spec_cmap   = lambda v: '#22c55e' if v>=0.9 else '#f59e0b' if v>=0.8 else '#ef4444'

bar_h(axes[0], df_plot.f1.values,          names, f'F1 Score (D2-F1+Recall)
Mean={df_plot.f1.mean():.3f}',           'F1',          vline=df_plot.f1.mean())
bar_h(axes[1], df_plot.recall.values,      names, f'Recall/Sensitivity (D2)
Mean={df_plot.recall.mean():.3f}',       'Recall',      vline=MIN_RECALL_CRITICAL, cmap_fn=recall_cmap)
bar_h(axes[2], df_plot.specificity.values, names, f'Specificity (D2)
Mean={df_plot.specificity.mean():.3f}',         'Specificity', vline=df_plot.specificity.mean(), cmap_fn=spec_cmap)

plt.suptitle('Cardiac AI V4 - Metrics tung benh (D2: F1+Recall-constrained)',
              color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/metrics_bar_v4.png', dpi=120, bbox_inches='tight', facecolor='#0f172a')
plt.show(); plt.close()
print('Saved: metrics_bar_v4.png')

## Cell 11 - Luu Thresholds & Bao cao CSV

In [ ]:
# ── Luu 3 bo thresholds ───────────────────────────────────────────────────
for name, thrs in [('youden',thr_youden),('f1rec',thr_f1rec),('screening',thr_screen)]:
    out = {str(i): float(t) for i,t in enumerate(thrs)}
    path = f'{OUTPUT_DIR}/thresholds/thr_{name}_v4.json'
    with open(path,'w') as f: json.dump(out, f, indent=2)
    print(f'Saved: thr_{name}_v4.json')

# ── Luu per-class CSV 3 dang ───────────────────────────────────────────────
for tag, df_ev in [('d1_youden',df_d1),('d2_f1rec',df_d2),('d3_screen',df_d3)]:
    df_ev.to_csv(f'{OUTPUT_DIR}/eval_{tag}_v4.csv', index=False)
    print(f'Saved: eval_{tag}_v4.csv')

# ── Bao cao tong ket (hoc tu Cell 53 trong Chest X-ray) ───────────────────
print(f'\n{"="*65}')
print(f'BAO CAO TONG KET — CARDIAC AI V4')
print(f'{"="*65}')
print(f'  Model   : Multi-scale CNN + Transformer (20M params)')
print(f'  Test set: {y_true_test.shape[0]:,} samples | TTA: 5x')
print(f'  Threshold step: 0.01 (chinh xac)')
print(f'  Benh nguy hiem: {[IDX_TO_NAME[i] for i in CRITICAL_DISEASES]} (Recall >= {MIN_RECALL_CRITICAL})')
print(f'')
print(f'  {"Chien luoc":<18} {"Macro F1":>9} {"Micro F1":>9} {"AUC":>7} {"mAP":>7}')
print(f'  {"-"*55}')
for tag, s in [("D1 Youden",s1),("D2 F1+Recall",s2),("D3 Screening",s3)]:
    print(f'  {tag:<18} {s["macro_f1"]:>9.4f} {s["micro_f1"]:>9.4f} {s["auc"]:>7.4f} {s["mAP"]:>7.4f}')
print(f'  {"="*55}')
print(f'')
print(f'  F1 >= 0.7 (D2): {(df_d2.f1>=0.7).sum()} classes')
print(f'  F1 >= 0.5 (D2): {(df_d2.f1>=0.5).sum()} classes')
print(f'  F1 <  0.3 (D2): {(df_d2.f1< 0.3).sum()} classes')
print(f'')
print(f'  KHUYEN NGHI SU DUNG:')
print(f'    Tieu chuan chung   → thr_youden_v4.json    (D1)')
print(f'    Y te, benh nguy   → thr_f1rec_v4.json     (D2) ← RECOMMENDED')
print(f'    Sang loc (screening)→ thr_screening_v4.json (D3)')
print(f'{"="*65}')

print(f'\nFILE OUTPUTS:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fp):
        print(f'  {f:<45} {os.path.getsize(fp)/1024:.1f} KB')
for f in sorted(os.listdir(f'{OUTPUT_DIR}/thresholds')):
    fp = os.path.join(OUTPUT_DIR,'thresholds',f)
    print(f'  thresholds/{f:<33} {os.path.getsize(fp)/1024:.1f} KB')